# Search Engine Pipeline v2

This notebook rebuilds the search index for CS papers, exports the Flask runtime artifacts, demonstrates the ranking pipeline, and evaluates the system against graded qrels.

Runtime-compatible outputs produced here:
- `vectorizer.pkl`
- `pos_weight_vector.pkl`
- `doc_ids.pkl`
- `doc_titles.pkl`
- `doc_abstracts.pkl`
- `doc_years.pkl`
- `doc_categories.pkl`
- `tfidf_base_matrix.npz`
- `tfidf_pos_matrix.npz`
- `click_store.pkl`
- `doc_authors.pkl` and `doc_author_tokens.pkl`

## 1. Environment Setup

This section loads the libraries, downloads the NLTK assets used by both indexing and query processing, and mounts Google Drive for Colab-based file access.

In [ ]:
!pip install rank-bm25 -q

import itertools
import pickle
import re
from collections import defaultdict

import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import scipy.sparse as sp
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

DRIVE_ROOT = '/content/drive/MyDrive'
RAW_DATA_PATH = f'{DRIVE_ROOT}/arxiv-metadata-oai-snapshot.json'
QRELS_PATH = f'{DRIVE_ROOT}/qrels.tsv'
ABSTRACT_SNIPPET_LENGTH = 300

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Load and Filter the CS Corpus

The indexing pipeline starts by reading the arXiv metadata snapshot in chunks, keeping only computer science papers, then filtering to documents with usable abstracts and recent publication years.

In [ ]:
chunks = []
for chunk in pd.read_json(RAW_DATA_PATH, lines=True, chunksize=10000):
    cs_chunk = chunk[chunk['categories'].str.contains(r'cs\.', na=False)]
    if len(cs_chunk) > 0:
        chunks.append(cs_chunk)
    if len(chunks) >= 50:
        break

df_cs = pd.concat(chunks, ignore_index=True)
print(f'Loaded {len(df_cs)} CS rows from the source snapshot')

In [ ]:
print('=== RAW DATA EXPLORATION ===')
print(f'Total rows loaded: {len(df_cs)}')
print(f'\nNull values:\n{df_cs.isnull().sum()}')
print(f"\nAbstract length stats:\n{df_cs['abstract'].str.len().describe()}")

df_cs_raw_year = df_cs[['id', 'title', 'abstract', 'update_date']].copy(deep=True)
df_cs_raw_year['year'] = df_cs_raw_year['update_date'].str[:4]
df_cs_raw_year['year'].value_counts().sort_index().plot(kind='bar', title='Papers per year (raw)')
plt.tight_layout()
plt.show()

In [ ]:
df_cs = df_cs[['id', 'title', 'abstract', 'authors', 'categories', 'update_date']].copy()
df_cs['year'] = df_cs['update_date'].str[:4]

before_count = len(df_cs)
df_cs = df_cs[df_cs['abstract'].str.len() >= 200]
after_abstract_filter = len(df_cs)
df_cs = df_cs[df_cs['year'].astype(int) >= 2015]
after_year_filter = len(df_cs)

print(f'Removed {before_count - after_abstract_filter} papers with abstracts under 200 characters')
print(f'Papers from 2015 onwards: {after_year_filter}')
print(f'Total CS papers after filtering: {len(df_cs)}')
print(f"Year range: {df_cs['year'].min()} - {df_cs['year'].max()}")

df_cs['year'].value_counts().sort_index().plot(kind='bar', title='Papers per year (filtered)')
plt.tight_layout()
plt.show()

## 3. Text Preparation and POS Weighting

The retrieval model uses a shared text normalization pipeline for both documents and queries. The title is repeated twice to give it more natural influence, while a separate POS-based weighting pass captures term importance before TF-IDF weighting is applied.

In [ ]:
def combine_fields(row):
    title = str(row['title']).strip()
    abstract = str(row['abstract']).strip()
    return f'{title} {title} {abstract}'

def remove_punctuation(text):
    text = re.sub(r'[^\w\s\-]', '', text)
    text = re.sub(r'\s-\s', ' ', text)
    return text

def preprocess_tokens(text):
    text = text.lower()
    text = remove_punctuation(text)
    text = re.sub(r'\d+', '', text)
    tokens = word_tokenize(text)
    tokens = [token for token in tokens if token not in stop_words and token.strip()]
    tokens = [stemmer.stem(token) for token in tokens]
    return tokens

def preprocess_text(text):
    return ' '.join(preprocess_tokens(text))

def get_pos_weights(text):
    tokens = word_tokenize(text.lower())
    tagged = nltk.pos_tag(tokens)

    weights = {}
    for word, tag in tagged:
        if '-' in word:
            weights[word] = 1.8
        elif tag.startswith('NN'):
            weights[word] = 2.0
        elif tag.startswith('VB'):
            weights[word] = 1.2
        elif tag.startswith('JJ'):
            weights[word] = 1.0
        else:
            weights[word] = 0.5
    return weights

def normalise_query(query):
    return ' '.join(sorted(preprocess_tokens(query)))

In [ ]:
df_cs['combined_text'] = df_cs.apply(combine_fields, axis=1)
df_cs['pos_weights'] = df_cs['combined_text'].apply(get_pos_weights)
df_cs['tokens'] = df_cs['combined_text'].apply(preprocess_tokens)
df_cs['processed_text'] = df_cs['tokens'].apply(lambda tokens: ' '.join(tokens))

print(df_cs[['title', 'processed_text']].head(3))
print('\nSample POS weights:')
print(dict(list(df_cs['pos_weights'].iloc[0].items())[:10]))

In [ ]:
df_cs.to_pickle(f'{DRIVE_ROOT}/df_cs_pos_tagged.pkl')
df_cs.to_pickle(f'{DRIVE_ROOT}/df_cs_cleaned.pkl')
print(f'Saved checkpoints for {len(df_cs)} cleaned documents')

In [ ]:
all_tokens_raw = list(itertools.chain(*df_cs['combined_text'].str.lower().str.split()))
all_tokens_clean = list(itertools.chain(*df_cs['tokens']))

vocab_before = len(set(all_tokens_raw))
vocab_after = len(set(all_tokens_clean))

plt.figure(figsize=(6, 4))
plt.bar(['Before cleaning', 'After cleaning'], [vocab_before, vocab_after], color=['#d9534f', '#5cb85c'])
plt.title('Vocabulary size reduction after ETL pipeline')
plt.ylabel('Unique terms')
for index, value in enumerate([vocab_before, vocab_after]):
    plt.text(index, value + 100, str(value), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('vocab_reduction.png', dpi=150)
plt.show()
print(f'Vocab before: {vocab_before}')
print(f'Vocab after:  {vocab_after}')
print(f'Reduction: {round((1 - vocab_after / vocab_before) * 100)}%')

## 4. Vectorization and POS-Weighted Index Construction

The base TF-IDF matrix is built from the cleaned corpus. A second matrix is derived by multiplying each vocabulary column by its average POS weight so the runtime can compare baseline and POS-aware ranking.

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=50000,
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 2)
)

tfidf_matrix = vectorizer.fit_transform(df_cs['processed_text'])
feature_names = vectorizer.get_feature_names_out()

print(f'Matrix shape: {tfidf_matrix.shape}')
print(f'Documents: {tfidf_matrix.shape[0]}, Features: {tfidf_matrix.shape[1]}')
print(f'Sample features: {feature_names[:20]}')

In [ ]:
def build_pos_weight_vector(feature_names, doc_pos_weights):
    term_weights = {}
    term_counts = {}

    for weight_map in doc_pos_weights:
        for term, weight in weight_map.items():
            stemmed_term = stemmer.stem(term)
            term_weights[stemmed_term] = term_weights.get(stemmed_term, 0.0) + weight
            term_counts[stemmed_term] = term_counts.get(stemmed_term, 0) + 1

    pos_weight_vector = np.ones(len(feature_names))
    for index, term in enumerate(feature_names):
        if term in term_weights:
            pos_weight_vector[index] = term_weights[term] / term_counts[term]

    return pos_weight_vector

pos_weight_vector = build_pos_weight_vector(feature_names, df_cs['pos_weights'])
pos_weight_diag = sp.diags(pos_weight_vector)
tfidf_pos_matrix = tfidf_matrix.dot(pos_weight_diag)

print(f'POS weight vector shape: {pos_weight_vector.shape}')
print(f'Min weight: {pos_weight_vector.min():.2f}')
print(f'Max weight: {pos_weight_vector.max():.2f}')
print(f'Mean weight: {pos_weight_vector.mean():.2f}')
print(f'POS-weighted matrix shape: {tfidf_pos_matrix.shape}')

## 5. Export Runtime Artifacts

This section writes the exact files expected by the Flask app. The document metadata lists must remain aligned with the TF-IDF matrix rows.

In [ ]:
def save_pickle(path, obj):
    with open(path, 'wb') as handle:
        pickle.dump(obj, handle)

sp.save_npz(f'{DRIVE_ROOT}/tfidf_pos_matrix.npz', tfidf_pos_matrix)
sp.save_npz(f'{DRIVE_ROOT}/tfidf_base_matrix.npz', tfidf_matrix)
save_pickle(f'{DRIVE_ROOT}/vectorizer.pkl', vectorizer)
save_pickle(f'{DRIVE_ROOT}/pos_weight_vector.pkl', pos_weight_vector)

doc_ids = df_cs['id'].tolist()
doc_titles = df_cs['title'].tolist()
doc_abstracts = df_cs['abstract'].tolist()
doc_years = pd.to_numeric(df_cs['year'], errors='coerce').fillna(0).astype(int).tolist()
doc_categories = df_cs['categories'].fillna('').astype(str).tolist()

save_pickle(f'{DRIVE_ROOT}/doc_ids.pkl', doc_ids)
save_pickle(f'{DRIVE_ROOT}/doc_titles.pkl', doc_titles)
save_pickle(f'{DRIVE_ROOT}/doc_abstracts.pkl', doc_abstracts)
save_pickle(f'{DRIVE_ROOT}/doc_years.pkl', doc_years)
save_pickle(f'{DRIVE_ROOT}/doc_categories.pkl', doc_categories)

print('Core index artifacts saved')

In [ ]:
def tokenise_authors(raw_authors):
    if not isinstance(raw_authors, str):
        return frozenset()

    parts = re.split(r'[,;]|\band\b', raw_authors, flags=re.IGNORECASE)
    tokens = set()
    for part in parts:
        for token in part.strip().lower().split():
            token = re.sub(r'[^a-z\-]', '', token)
            if token and len(token) > 1:
                tokens.add(stemmer.stem(token))
    return frozenset(tokens)

doc_authors = df_cs['authors'].tolist()
doc_author_tokens = [tokenise_authors(author) for author in doc_authors]
save_pickle(f'{DRIVE_ROOT}/doc_authors.pkl', doc_authors)
save_pickle(f'{DRIVE_ROOT}/doc_author_tokens.pkl', doc_author_tokens)

sample_tokens = list(doc_author_tokens[0])[:5] if doc_author_tokens else []
print(f'Author artifacts saved: {len(doc_authors)} docs, sample tokens: {sample_tokens}')

In [ ]:
files = [
    'vectorizer.pkl', 'pos_weight_vector.pkl', 'doc_ids.pkl',
    'doc_titles.pkl', 'doc_abstracts.pkl', 'doc_years.pkl', 'doc_categories.pkl',
    'tfidf_pos_matrix.npz', 'tfidf_base_matrix.npz',
    'doc_authors.pkl', 'doc_author_tokens.pkl'
]
for filename in files:
    path = f'{DRIVE_ROOT}/{filename}'
    try:
        with open(path, 'rb'):
            pass
        print(f'✓ {filename}')
    except FileNotFoundError:
        print(f'✗ {filename}')

## 6. Search Functions and Comparison Checks

The search functions all share the same preprocessing helper. This reduces duplication while keeping the retrieval logic identical to the original notebook and the Flask runtime.

In [ ]:
def build_query_vector(query, use_pos_weights=True):
    query_processed = preprocess_text(query)
    query_vector = vectorizer.transform([query_processed])
    if use_pos_weights:
        return query_vector.dot(pos_weight_diag)
    return query_vector

def make_search_result(index, rank, score, score_key='score'):
    return {
        'rank': rank,
        'id': doc_ids[index],
        'title': doc_titles[index],
        'abstract': doc_abstracts[index][:ABSTRACT_SNIPPET_LENGTH] + '...',
        score_key: round(float(score), 4)
    }

def search(query, top_k=10):
    query_vector = build_query_vector(query, use_pos_weights=True)
    scores = cosine_similarity(query_vector, tfidf_pos_matrix).flatten()
    top_indices = scores.argsort()[::-1][:top_k]
    return [make_search_result(index, rank, scores[index]) for rank, index in enumerate(top_indices, 1)]

def search_baseline(query, top_k=10):
    query_vector = build_query_vector(query, use_pos_weights=False)
    scores = cosine_similarity(query_vector, tfidf_matrix).flatten()
    top_indices = scores.argsort()[::-1][:top_k]
    return [make_search_result(index, rank, scores[index]) for rank, index in enumerate(top_indices, 1)]

In [ ]:
demo_query = 'deep learning image classification'
print(f'Demo query: {demo_query}')
for result in search(demo_query, top_k=5):
    print(f"{result['rank']}. [{result['score']}] {result['title']}")
    print(f"    {result['abstract'][:150]}...")
    print()

comparison_query = 'neural network optimisation'
print(f"Query: '{comparison_query}'\n")
print('=== BASELINE TF-IDF ===')
for result in search_baseline(comparison_query, top_k=5):
    print(f"  {result['rank']}. [{result['score']}] {result['title']}")

print('\n=== POS-WEIGHTED TF-IDF ===')
for result in search(comparison_query, top_k=5):
    print(f"  {result['rank']}. [{result['score']}] {result['title']}")

In [ ]:
test_queries = [
    'deep learning image classification',
    'neural network optimisation',
    'natural language processing transformer',
    'reinforcement learning robotics',
    'graph neural network node classification',
    'convolutional network object detection'
]

changes = 0
for query in test_queries:
    baseline_title = search_baseline(query, top_k=1)[0]['title']
    pos_title = search(query, top_k=1)[0]['title']
    if baseline_title != pos_title:
        changes += 1
        print(f"CHANGED: '{query}'")
        print(f'  Baseline: {baseline_title[:60]}')
        print(f'  POS:      {pos_title[:60]}')
        print()

print(f'Top result changed in {changes}/{len(test_queries)} queries ({round(changes / len(test_queries) * 100)}%)')

## 7. Click-Through Re-Ranking

The click store records user feedback per normalized query. Search results can then blend retrieval scores with normalized click evidence using the same alpha-controlled weighting as the current runtime.

In [ ]:
click_store = defaultdict(lambda: defaultdict(int))

def record_click(query, doc_id):
    query_key = normalise_query(query)
    click_store[query_key][doc_id] += 1
    print(f"Recorded click: query='{query_key}' doc='{doc_id}'")

def get_click_score(query, doc_id):
    query_key = normalise_query(query)
    clicks = click_store[query_key]
    if not clicks or doc_id not in clicks:
        return 0.0
    max_clicks = max(clicks.values())
    return clicks[doc_id] / max_clicks

def search_with_clicks(query, top_k=10, alpha=0.7):
    query_vector = build_query_vector(query, use_pos_weights=True)
    cosine_scores = cosine_similarity(query_vector, tfidf_pos_matrix).flatten()

    blended_scores = []
    for index in range(len(doc_ids)):
        doc_id = doc_ids[index]
        cosine_score = cosine_scores[index]
        click_score = get_click_score(query, doc_id)
        blended_scores.append(alpha * cosine_score + (1 - alpha) * click_score)

    blended_scores = np.array(blended_scores)
    top_indices = blended_scores.argsort()[::-1][:top_k]

    results = []
    for rank, index in enumerate(top_indices, 1):
        doc_id = doc_ids[index]
        click_score = get_click_score(query, doc_id)
        results.append({
            'rank': rank,
            'id': doc_id,
            'title': doc_titles[index],
            'abstract': doc_abstracts[index][:ABSTRACT_SNIPPET_LENGTH] + '...',
            'cosine_score': round(float(cosine_scores[index]), 4),
            'click_score': round(float(click_score), 4),
            'blended_score': round(float(blended_scores[index]), 4)
        })

    return results

print('Click store initialised')

In [ ]:
click_demo_query = 'graph neural network node classification'

print('=== BEFORE ANY CLICKS ===')
results_before = search_with_clicks(click_demo_query, top_k=5)
for result in results_before:
    print(f"  {result['rank']}. [{result['blended_score']}] {result['title'][:60]}")

clicked_doc_id = results_before[2]['id']
clicked_doc_title = results_before[2]['title']
record_click(click_demo_query, clicked_doc_id)
record_click(click_demo_query, clicked_doc_id)
print(f"\nSimulated 2 clicks on: '{clicked_doc_title[:60]}'")

print('\n=== AFTER CLICKS — same query ===')
results_after = search_with_clicks(click_demo_query, top_k=5)
for result in results_after:
    marker = ' <-- boosted' if result['id'] == clicked_doc_id else ''
    print(f"  {result['rank']}. [{result['blended_score']}] (cos:{result['cosine_score']} | click:{result['click_score']}) {result['title'][:50]}{marker}")

In [ ]:
print(f"{'Alpha':<8} {'Rank-1 Title':<55} {'Boosted doc rank'}")
print('-' * 80)
for alpha in [0.5, 0.6, 0.7, 0.8, 0.9]:
    top_results = search_with_clicks(click_demo_query, top_k=10, alpha=alpha)
    rank1_title = top_results[0]['title'][:50].replace('\\n', ' ')
    boosted_rank = next((result['rank'] for result in top_results if result['id'] == clicked_doc_id), 'N/A')
    print(f"{alpha:<8} {rank1_title:<55} {boosted_rank}")

In [ ]:
save_pickle(f'{DRIVE_ROOT}/click_store.pkl', dict(click_store))
print(f'Click store saved with {len(click_store)} query entries')

## 8. Evaluation Against Graded Qrels

The final section evaluates three systems on the same query set using graded relevance labels: TF-IDF baseline, TF-IDF (POS-weighted), and BM25. Metrics: P@5, P@10, Recall@10, AP, RR, and NDCG@10.

In [ ]:
qrels = defaultdict(dict)
with open(QRELS_PATH, 'r', encoding='utf-8') as handle:
    next(handle)
    for line in handle:
        parts = line.strip().split('\t')
        if len(parts) == 3:
            query_id, doc_id, grade = parts[0], parts[1], int(parts[2])
            qrels[query_id][doc_id] = grade

eval_queries = {
    'Q0': 'deep learning image classification',
    'Q1': 'neural network optimisation',
    'Q2': 'natural language processing transformer',
    'Q3': 'reinforcement learning robotics',
    'Q4': 'graph neural network node classification',
    'Q5': 'convolutional network object detection',
}

def precision_at_k(ranked, relevant, k):
    return sum(1 for doc_id in ranked[:k] if relevant.get(doc_id, 0) >= 1) / k

def recall_at_k(ranked, relevant, k):
    relevant_total = sum(1 for grade in relevant.values() if grade >= 1)
    if not relevant_total:
        return 0.0
    return sum(1 for doc_id in ranked[:k] if relevant.get(doc_id, 0) >= 1) / relevant_total

def average_precision(ranked, relevant):
    relevant_total = sum(1 for grade in relevant.values() if grade >= 1)
    if not relevant_total:
        return 0.0

    hits = 0
    total = 0.0
    for index, doc_id in enumerate(ranked, 1):
        if relevant.get(doc_id, 0) >= 1:
            hits += 1
            total += hits / index
    return total / relevant_total

def reciprocal_rank(ranked, relevant):
    for index, doc_id in enumerate(ranked, 1):
        if relevant.get(doc_id, 0) >= 1:
            return 1.0 / index
    return 0.0

def dcg(ranked, relevant, k):
    return sum(relevant.get(doc_id, 0) / np.log2(index + 1) for index, doc_id in enumerate(ranked[:k], 1))

def ndcg_at_k(ranked, relevant, k):
    ideal = sorted((grade for grade in relevant.values() if grade > 0), reverse=True)
    idcg = sum(grade / np.log2(index + 1) for index, grade in enumerate(ideal[:k], 1))
    if not idcg:
        return 0.0
    return dcg(ranked, relevant, k) / idcg

def evaluate_run(results, qrels_data, k=10):
    rows = []
    for query_id, ranked in results.items():
        relevant = qrels_data.get(query_id, {})
        rows.append({
            'qid': query_id,
            'P@5': precision_at_k(ranked, relevant, 5),
            'P@10': precision_at_k(ranked, relevant, k),
            'Recall@10': recall_at_k(ranked, relevant, k),
            'AP': average_precision(ranked, relevant),
            'RR': reciprocal_rank(ranked, relevant),
            'NDCG@10': ndcg_at_k(ranked, relevant, k)
        })
    return rows

METRICS = ['P@5', 'P@10', 'Recall@10', 'AP', 'RR', 'NDCG@10']
print(f'qrels loaded from {QRELS_PATH}: {sum(len(value) for value in qrels.values())} judgments across {len(qrels)} queries')
corpus_id_set = {str(d) for d in doc_ids}
qrels_ids = {did for rel in qrels.values() for did in rel}
print(f'ID alignment: {len(qrels_ids & corpus_id_set)}/{len(qrels_ids)} qrels docs found in corpus')

In [ ]:
base_results = {}
for query_id, query in eval_queries.items():
    query_vector = build_query_vector(query, use_pos_weights=False)
    scores = cosine_similarity(query_vector, tfidf_matrix).flatten()
    top_indices = scores.argsort()[::-1][:10]
    base_results[query_id] = [str(doc_ids[i]) for i in top_indices]

base_rows = evaluate_run(base_results, qrels)
means_base = {m: np.mean([r[m] for r in base_rows]) for m in METRICS}
print('--- TF-IDF (baseline, no POS weighting) ---')
print(f"{'Query':<8}" + ''.join(f"{metric:>12}" for metric in METRICS))
print('-' * 80)
for row in base_rows:
    print(f"{row['qid']:<8}" + ''.join(f"{row[metric]:>12.4f}" for metric in METRICS))
print('-' * 80)
print(f"{'MEAN':<8}" + ''.join(f"{means_base[metric]:>12.4f}" for metric in METRICS))

tfidf_results = {}
for query_id, query in eval_queries.items():
    query_vector = build_query_vector(query, use_pos_weights=True)
    scores = cosine_similarity(query_vector, tfidf_pos_matrix).flatten()
    top_indices = scores.argsort()[::-1][:10]
    tfidf_results[query_id] = [str(doc_ids[index]) for index in top_indices]

tfidf_rows = evaluate_run(tfidf_results, qrels)
print('--- TF-IDF Cosine (POS-weighted) ---')
print(f"{'Query':<8}" + ''.join(f"{metric:>12}" for metric in METRICS))
print('-' * 80)
for row in tfidf_rows:
    print(f"{row['qid']:<8}" + ''.join(f"{row[metric]:>12.4f}" for metric in METRICS))
print('-' * 80)
means_tfidf = {metric: np.mean([row[metric] for row in tfidf_rows]) for metric in METRICS}
print(f"{'MEAN':<8}" + ''.join(f"{means_tfidf[metric]:>12.4f}" for metric in METRICS))

In [ ]:
from rank_bm25 import BM25Okapi

print('Tokenizing corpus for BM25...')
bm25_corpus = [
    preprocess_tokens(str(doc_titles[index]) * 2 + ' ' + str(doc_abstracts[index]))
    for index in range(len(doc_ids))
]
bm25_index = BM25Okapi(bm25_corpus)

bm25_results = {}
for query_id, query in eval_queries.items():
    query_tokens = preprocess_tokens(query)
    scores = bm25_index.get_scores(query_tokens)
    top_indices = scores.argsort()[::-1][:10]
    bm25_results[query_id] = [str(doc_ids[index]) for index in top_indices]

bm25_rows = evaluate_run(bm25_results, qrels)
print('\n--- BM25 (rank-bm25, Okapi) ---')
print(f"{'Query':<8}" + ''.join(f"{metric:>12}" for metric in METRICS))
print('-' * 80)
for row in bm25_rows:
    print(f"{row['qid']:<8}" + ''.join(f"{row[metric]:>12.4f}" for metric in METRICS))
print('-' * 80)
means_bm25 = {metric: np.mean([row[metric] for row in bm25_rows]) for metric in METRICS}
print(f"{'MEAN':<8}" + ''.join(f"{means_bm25[metric]:>12.4f}" for metric in METRICS))

In [ ]:
summary = {
    'Metric': METRICS,
    'TF-IDF (baseline)': [round(means_base[m], 4) for m in METRICS],
    'TF-IDF (POS-weighted)': [round(means_tfidf[m], 4) for m in METRICS],
    'BM25 (Okapi)': [round(means_bm25[m], 4) for m in METRICS],
}

comparison_df = pd.DataFrame(summary)
print('\n=== System Comparison (macro-averaged over 6 queries) ===')
print(comparison_df.to_string(index=False))
print(
    f"\nMAP - baseline: {means_base['AP']:.4f} | "
    f"POS-weighted: {means_tfidf['AP']:.4f} | "
    f"BM25: {means_bm25['AP']:.4f}"
)

## 9. Final Notes

This v2 notebook keeps the existing functionality but presents it in a clearer top-to-bottom workflow. The main compatibility requirements are unchanged: preprocessing parity with the Flask app, stable vectorizer settings, and exact artifact filenames.